In [7]:
import neuro
import caspian
from rss.CasPyanBinaryController import CasPyanBinaryController
from rss.CasPyanBinaryRemappedController import CasPyanBinaryRemappedController
from rss.CaspianBinaryRemappedController import CaspianBinaryRemappedController
from swarmsim.agent.MazeAgent import MazeAgent, MazeAgentConfig
from swarmsim.world.RectangularWorld import RectangularWorld, RectangularWorldConfig
from swarmsim.sensors.BinaryFOVSensor import BinaryFOVSensor
import math
import numpy as np

In [8]:
network = neuro.Network()
print(dir(network))

['__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '_pybind11_conduit_v1_', 'add_edge', 'add_input', 'add_node', 'add_or_get_edge', 'add_or_get_node', 'add_output', 'as_json', 'assign', 'clear', 'data_keys', 'edges', 'from_json', 'from_str', 'get_data', 'get_edge', 'get_edge_property', 'get_input', 'get_network_property', 'get_node', 'get_node_property', 'get_output', 'get_properties', 'get_random_edge', 'get_random_input', 'get_random_node', 'get_random_ouptut', 'is_edge', 'is_edge_property', 'is_network_property', 'is_node', 'is_node_property', 'make_sorted_node_vector', 'nodes', 'num_edges', 'num_inputs', 'num_nodes', 'num_outputs', 'pretty_edges', 'pretty_json', 

In [11]:
network_path = "/home/shan/neuromorphic/turtwig/results_sim/hopper/250804/tenn2/n-6/250729-132733-connorsim_snn_eons-v01/best.json" 

# Load network from file
network = neuro.Network()
network.read_from_file(network_path)

# Create controller
controller = CaspianBinaryRemappedController(agent=None, network=network, neuro_tpc=1)

# Create world and agent
world_config = RectangularWorldConfig(size=(10, 10), time_step=1/40)
world = RectangularWorld(world_config)
agent = MazeAgent(MazeAgentConfig(position=(4, 4), agent_radius=0.1), world)
world.population.append(agent)

# Adding the sensor
sensor = BinaryFOVSensor(agent, theta=0.45, distance=1.0)
agent.sensors.append(sensor)

agent.set_heading(-2.119)

controller.set_agent(agent)
agent.controller = controller

In [12]:
input_array = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

# Run simulation
dt = 1/40
results = []

for timestep, sensor_input in enumerate(input_array):
    sensor.current_state = bool(sensor_input)
    v, w = controller.run_processor(bool(sensor_input))
    
    heading = agent.get_heading()
    x = agent.get_x_pos() + v * dt * math.cos(heading)
    y = agent.get_y_pos() + v * dt * math.sin(heading)
    new_heading = heading + w * dt
    
    agent.set_x_pos(x)
    agent.set_y_pos(y)
    agent.set_heading(new_heading)
    
    results.append({
        'timestep': timestep,
        'input': sensor_input,
        'velocity': v,
        'angular_velocity': w,
        'x': x,
        'y': y,
        'heading': new_heading
    })

# Convert to DataFrame for easy analysis
import pandas as pd
df = pd.DataFrame(results)
df

[0, 0, 0, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]
[0, 0, 1, 0]


,timestep,input,velocity,angular_velocity,x,y,heading
0,0,0,0.276,-0.602,3.996404,3.994111,-2.13405
1,1,0,-0.276,-0.602,4.000088,3.999945,-2.14910
2,2,0,-0.276,-0.602,4.003860,4.005723,-2.16415
3,3,0,0.276,-0.602,4.000002,4.000003,-2.17920
4,4,0,-0.276,-0.602,4.003945,4.005665,-2.19425
5,5,0,-0.276,-0.602,4.007974,4.011266,-2.20930
6,6,0,0.276,-0.602,4.003862,4.005726,-2.22435
7,7,0,0.276,-0.602,3.999666,4.000248,-2.23940
8,8,0,0.276,-0.602,3.995389,3.994833,-2.25445
9,9,0,0.276,-0.602,3.991031,3.989484,-2.26950


In [6]:
print(f"Agent initial position: {agent.get_x_pos()}, {agent.get_y_pos()}")
print(f"Agent initial heading: {agent.get_heading()}")
print(f"Sensor distance: {sensor.distance}")
print(f"Sensor theta: {sensor.theta}")
print(f"neuro_tpc: {controller.neuro_tpc}")

Agent initial position: 5.068812555556031, 5.005394963008391
Agent initial heading: 0.09030000000000002


AttributeError: 'BinaryFOVSensor' object has no attribute 'distance'